# SpatialGDC on DLPFC
This tutorial demonstrates spatial domain identification on the DLPFC dataset (Visium, 12 slices).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys; sys.path.insert(0, "..")
from spatialgdc import SpatialGDC, prepare_graph, compute_spatial_keep_prob, clustering, fix_seed
from sklearn.decomposition import PCA

fix_seed(0)

## 1. Load and Preprocess Data

In [ ]:
# Load Visium data
sample = "151507"
data_path = f"../dataset/DLPFC/{sample}/"
adata = sc.read_visium(data_path)
adata.var_names_make_unique()
adata.obs_names = adata.obs_names.str.replace("-1", "", regex=False).str.strip()

# Load ground truth labels
meta = pd.read_csv(f"{data_path}/metadata.tsv", sep="\t", index_col=0)
adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess"]
n_clusters = 5 if sample in ["151669","151670","151671","151672"] else 7

# Preprocessing
sc.pp.filter_genes(adata, min_counts=1)
sc.pp.filter_cells(adata, min_counts=1)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata)
adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)

print(f"Spots: {adata.shape[0]}, Genes: {adata.shape[1]}, Clusters: {n_clusters}")

## 2. Build Dual Graphs

In [ ]:
g_spatial = prepare_graph(adata, "spatial")
g_expr = prepare_graph(adata, "expr")
graph_dict = {"spatial": g_spatial, "expr": g_expr}

# Spatial prior for adaptive edge dropout
coords = adata.obsm["spatial"].copy()
coords = (coords - coords.min(axis=0)) / (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
keep_prob = compute_spatial_keep_prob(g_expr, coords, sigma=0.4)

print(f"Spatial edges: {g_spatial.nnz}, Expr edges: {g_expr.nnz}")

## 3. Train SpatialGDC

In [ ]:
model = SpatialGDC(
    input_data=adata.obsm["X_pca"].copy(),
    graph_dict=graph_dict,
    n_clusters=n_clusters,
    expr_keep_prob=keep_prob,
    gamma=1.0, kappa=0.1,
    use_spatial_drop=True,
    use_intersection_cl=True,
)
pred_labels, embeddings, x_rec = model.train()
print(f"Embedding shape: {embeddings.shape}")

## 4. Cluster and Evaluate

In [ ]:
adata.obsm["emb"] = embeddings
clustering(adata, n_clusters, key="emb", refinement=True, cluster_methods="mclust")

# Evaluate
from sklearn.metrics import adjusted_rand_score
cluster_col = "mclust_refined" if "mclust_refined" in adata.obs.columns else "mclust"
ari = adjusted_rand_score(adata.obs["Region"], adata.obs[cluster_col])
print(f"ARI = {ari:.4f}")

## 5. Visualize

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.spatial(adata, color="Region", ax=axes[0], show=False, title="Ground Truth")
sc.pl.spatial(adata, color=cluster_col, ax=axes[1], show=False, title=f"SpatialGDC (ARI={ari:.3f})")
plt.tight_layout()
plt.show()